In [17]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [20]:
path_data = 'dataraw/payments.csv'
data = pd.read_csv(path_data)
data.head(5)

,payment_id,invoice_id,amount_paid
0,PMT0001,INV0001,220000000.0
1,PMT0002,INV0002,209000000.0
2,PMT0003,INV0003,94600000.0
3,PMT0004,INV0004,269500000.0
4,PMT0005,INV0005,267300000.0


## MySQL (pymysql)

Uses the `DB_*` settings in `.env` (Docker container `loitph-mysql`, port 3306).

In [26]:
import pymysql

def mysql_connect(database=None):
    return pymysql.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=database,
        cursorclass=pymysql.cursors.DictCursor,  # rows as dicts, like as_dict=True in pymssql
        autocommit=True,
    )

# MySQL starts empty, so recreate the Employees table from SQL Server
with mysql_connect(os.getenv("DB_NAME")) as conn, conn.cursor() as cursor:
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS Employees (
            EmployeeID INT PRIMARY KEY,
            FirstName  VARCHAR(50),
            LastName   VARCHAR(50),
            HireDate   DATE
        );
    """)
    cursor.executemany(
        "INSERT IGNORE INTO Employees VALUES (%s, %s, %s, %s);",  # IGNORE skips rows that already exist
        [
            (1, "John", "Doe", "2020-01-15"),
            (2, "Jane", "Smith", "2019-03-22"),
            (3, "Michael", "Johnson", "2021-07-30"),
            (4, "Emily", "Davis", "2018-11-05"),
            (5, "William", "Brown", "2022-02-14"),
        ],
    )
    print("Employees table ready.")

Employees table ready.


In [27]:
with mysql_connect(os.getenv("DB_NAME")) as conn, conn.cursor() as cursor:
    cursor.execute("SELECT * FROM Employees;")
    df = pd.DataFrame(cursor.fetchall())

print(df)

   EmployeeID FirstName LastName    HireDate
0           1      John      Doe  2020-01-15
1           2      Jane    Smith  2019-03-22
2           3   Michael  Johnson  2021-07-30
3           4     Emily    Davis  2018-11-05
4           5   William    Brown  2022-02-14


In [28]:
# create database
# MySQL supports IF NOT EXISTS directly, no DB_ID() check needed
mysql_new_db = os.getenv("NEW_DB_NAME")

with mysql_connect() as conn, conn.cursor() as cursor:
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS `{mysql_new_db}`;")
    print(f"Database '{mysql_new_db}' is ready.")

Database 'SqlPrimer2' is ready.


In [25]:
# Create a SQLAlchemy engine for the new MySQL database
# URL.create escapes special characters in the password (@, :, / would break an f-string URL)
from sqlalchemy.engine import URL

mysql_url = URL.create(
    "mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=mysql_new_db,
)
mysql_engine = create_engine(mysql_url)

with mysql_engine.connect() as conn:
    print(conn.execute(text("SELECT DATABASE();")).scalar())

SqlPrimer2


In [31]:
new_table_name = "payments"

try:
    data.to_sql(
      new_table_name,
      con=mysql_engine,
      if_exists="replace",
      index=False
    )
    print(f"Table '{new_table_name}' created in database '{mysql_new_db}'.")
except Exception as e:
    print(f"Error creating table '{new_table_name}': {e}")
    raise

Table 'payments' created in database 'SqlPrimer2'.


In [32]:
customers_data = pd.read_csv('dataraw/customers.csv')
table_name = "customers"

try:
    data.to_sql(
      table_name,
      con=mysql_engine,
      if_exists="replace",
      index=False
    )
    print(f"Table '{table_name}' created in database '{mysql_new_db}'.")
except Exception as e:
    print(f"Error creating table '{table_name}': {e}")
    raise

Table 'customers' created in database 'SqlPrimer2'.
